# 🏥 Sympriority — Voice-to-Triage Pipeline (Ministral-3B Edition)

**Completely standalone notebook — run all cells top-to-bottom on Google Colab (free T4 GPU).**

| Stage | Model | VRAM |
|---|---|---|
| 🎤 Speech  | `openai/whisper-medium` | ~2 GB |
| 🤖 Symptoms  | `mistralai/Ministral-3-3B-Instruct-2512` | ~6 GB |
| **Total** | — | **~9 GB** (T4 has 15 GB) |

**Features:**
- Speaks any language (Hindi, Tamil, Telugu, English) → auto-translated to English by Whisper
- 4-level clinical triage: Critical / High / Moderate / Low
- Color-coded triage card with priority score, recommended action, and reasoning
- Gradio UI with `share=True` link (works from Colab)

In [ ]:
#cell 1
!pip install -q transformers accelerate datasets librosa soundfile


In [ ]:
#cell 2
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset
from IPython.display import Audio, display

print("CUDA:", torch.cuda.is_available())

In [ ]:
#cell 3
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print("Using:", device)

In [ ]:
#cell 4
model_id = "vasista22/whisper-hindi-large-v2"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True
).to(device)

processor = AutoProcessor.from_pretrained(model_id)

print("Model loaded ✅")

In [ ]:
#cell 5

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=0 if device == "cuda:0" else -1,
)
print("Pipeline ready ✅")

In [ ]:
#cell 6
from datasets import load_dataset
from IPython.display import Audio, display

print("Loading sample audio...")

dataset = load_dataset("hf-internal-testing/librispeech_asr_dummy", split="validation")

sample = dataset[0]

# Play audio
display(Audio(sample["audio"]["array"], rate=sample["audio"]["sampling_rate"]))

print("Original Text:")
print(sample["text"])

In [ ]:
#cell 7
# Get the forced_decoder_ids first, which includes the language and task tokens
forced_decoder_ids_val = processor.get_decoder_prompt_ids(language="hi", task="transcribe")

# Extract the token ID for 'hi' from the forced_decoder_ids
# The language token is typically the second element in the forced_decoder_ids list
# e.g., [(1, start_token_id), (2, language_token_id), (3, task_token_id)]
lang_id_for_hi = forced_decoder_ids_val[1][1]

# Set lang_to_id on generation_config using the extracted language ID
model.generation_config.lang_to_id = {"hi": lang_id_for_hi}

# Set the forced_decoder_ids on generation_config
model.generation_config.forced_decoder_ids = forced_decoder_ids_val

predicted_ids = model.generate(
    input_features.to(device)
)

transcription = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0]

print("Clean Output:")
print(transcription.strip())


In [ ]:
#cell 8
# 🎤 Record + Transcribe in ONE CELL

from IPython.display import Javascript, display
from google.colab import output
import base64
import librosa

# Step 1: Record Audio
def record_audio(seconds=5):
    display(Javascript(f"""
    async function record() {{
        const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
        const recorder = new MediaRecorder(stream);
        let chunks = [];

        recorder.ondataavailable = e => chunks.push(e.data);
        recorder.start();

        await new Promise(resolve => setTimeout(resolve, {seconds} * 1000));

        recorder.stop();
        await new Promise(resolve => recorder.onstop = resolve);

        const blob = new Blob(chunks);
        const arrayBuffer = await blob.arrayBuffer();
        const base64String = btoa(
            new Uint8Array(arrayBuffer)
                .reduce((data, byte) => data + String.fromCharCode(byte), '')
        );

        return base64String;
    }}
    record();
    """))

    audio_base64 = output.eval_js("record()")
    audio_bytes = base64.b64decode(audio_base64)

    with open("recorded.wav", "wb") as f:
        f.write(audio_bytes)

    return "recorded.wav"

# Step 2: Record
print("🎤 Recording... Speak now!")
audio_file = record_audio(5)
print("✅ Recording complete!")

# Step 3: Load Audio
audio_array, sr = librosa.load(audio_file, sr=16000)

# Step 4: Convert to model input
input_features = processor(
    audio_array,
    sampling_rate=sr,
    return_tensors="pt"
).input_features.to(torch_dtype)

# Step 5: Transcribe + Translate (Hindi → English)
# Get the forced_decoder_ids for the target language 'en' and task 'translate'
forced_decoder_ids_en_translate = processor.get_decoder_prompt_ids(language="en", task="translate")

# Extract the token ID for 'en' (English) and 'translate' task
lang_id_for_en = forced_decoder_ids_en_translate[1][1] # Typically the language token is the second element
task_id_for_translate = forced_decoder_ids_en_translate[2][1] # And task token the third

# Step 5: Hindi Speech → English Text

predicted_ids = model.generate(
    input_features.to(device),
    task="translate",   #  THIS enables translation
    language="en"       #  Output in English
)

transcription = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0]

# print("\n🎤 Translated Output:")
# print(transcription.strip())

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load the translation model and tokenizer directly
translator_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-hi-en")
translator_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-hi-en")

# Encode the Hindi transcription
inputs = translator_tokenizer(transcription, return_tensors="pt")

# Generate translation
translated_tokens = translator_model.generate(**inputs)

# Decode to English text
translated = translator_tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

print("Translated:", translated)


In [ ]:
#cell 9
# ============================================================
# 🌍 FULL PIPELINE — Multilingual Speech → English Text
# Supports: English, Hindi (हिन्दी), Telugu (తెలుగు), Tamil (தமிழ்)
# Uses whisper-medium — GPU-safe on free Colab T4 (~2 GB VRAM)
# ✅ STANDALONE — no dependency on previous cells
# ============================================================

# ── USER CONFIG — set BEFORE running ─────────────────────────
#   "en" = English | "hi" = Hindi | "te" = Telugu | "ta" = Tamil
LANGUAGE       = "en"   # ← CHANGE THIS to your spoken language
RECORD_SECONDS = 7      # ← recording duration in seconds
# ─────────────────────────────────────────────────────────────

# ── Step 0: Dependencies ──────────────────────────────────────
!apt-get install -y -q ffmpeg
!pip install -q openai-whisper

import torch, base64, whisper
from IPython.display import Javascript, display
from google.colab import output

LANG_NAMES   = {"en": "English", "hi": "Hindi", "te": "Telugu", "ta": "Tamil"}
lang_display = LANG_NAMES.get(LANGUAGE, LANGUAGE.upper())
device       = "cuda" if torch.cuda.is_available() else "cpu"

print(f"✅ Device       : {device.upper()}")
print(f"🌐 Language set : {lang_display} ({LANGUAGE})")

# ── Step 1: Load whisper-medium ───────────────────────────────
# ~1.4 GB — downloads once, cached for the session
# Uses ~2 GB VRAM on T4 (well within free tier limit)
print("\n⏳ Loading whisper-medium ...")
asr_model = whisper.load_model("medium", device=device)
print("✅ whisper-medium loaded")

# ── Step 2: Record via browser mic ───────────────────────────
def record_audio(duration_sec, filename="patient_input.webm"):
    js_code = f"""
    async function recordAudio() {{
        const stream   = await navigator.mediaDevices.getUserMedia({{ audio: true }});
        const mimeType = MediaRecorder.isTypeSupported('audio/webm;codecs=opus')
                         ? 'audio/webm;codecs=opus' : 'audio/webm';
        const recorder = new MediaRecorder(stream, {{ mimeType }});
        const chunks   = [];
        recorder.ondataavailable = e => chunks.push(e.data);
        recorder.start();
        await new Promise(r => setTimeout(r, {duration_sec * 1000}));
        recorder.stop();
        stream.getTracks().forEach(t => t.stop());
        await new Promise(r => recorder.onstop = r);
        const blob   = new Blob(chunks, {{ type: mimeType }});
        const buffer = await blob.arrayBuffer();
        const bytes  = new Uint8Array(buffer);
        const chunk  = 8192;
        let   b64    = '';
        for (let i = 0; i < bytes.length; i += chunk)
            b64 += String.fromCharCode.apply(null, bytes.subarray(i, i + chunk));
        return btoa(b64);
    }}
    recordAudio();
    """
    display(Javascript(js_code))
    audio_bytes = base64.b64decode(output.eval_js("recordAudio()"))
    with open(filename, "wb") as f:
        f.write(audio_bytes)
    size_kb = len(audio_bytes) / 1024
    print(f"✅ Audio saved → {filename}  ({size_kb:.1f} KB)")
    if size_kb < 5:
        print("⚠️  File is very small — check browser mic permission and re-run")
    return filename

print(f"\n🎤 Speak now in {lang_display} — {RECORD_SECONDS} seconds ...")
audio_path = record_audio(RECORD_SECONDS)

# ── Step 3: Transcribe with explicit language hint ────────────
# CRITICAL: always pass language= explicitly.
# Without it, whisper-medium silently returns empty text for Telugu/Tamil.
print("\n🔍 Transcribing ...")
result = asr_model.transcribe(
    audio_path,
    task="transcribe",
    language=LANGUAGE,          # explicit — prevents silent empty output
    beam_size=3,                # 3 is faster than default 5, still accurate
    fp16=(device == "cuda")     # float16 on GPU → halves VRAM usage
)
original_text = result["text"].strip()

if not original_text:
    print("⚠️  Transcription empty — mic may not have captured audio.")
    print("   1. Allow mic permission in the browser banner and re-run")
    print("   2. Speak louder/closer to mic")
else:
    print(f"📝 {lang_display} Text:\n   {original_text}")

# ── Step 4: Translate to English ─────────────────────────────
if LANGUAGE == "en" or not original_text:
    final_english = original_text
    if LANGUAGE == "en":
        print("\n✅ Already English — no translation needed.")
else:
    print(f"\n🔄 Translating {lang_display} → English ...")
    t_result = asr_model.transcribe(
        audio_path,
        task="translate",
        language=LANGUAGE,
        beam_size=3,
        fp16=(device == "cuda")
    )
    final_english = t_result["text"].strip()

# ── Step 5: Output ────────────────────────────────────────────
print("\n" + "═" * 55)
print("🎯  FINAL ENGLISH OUTPUT")
print("═" * 55)
print(final_english or "(empty — re-check microphone and retry)")
print("═" * 55)


In [ ]:
#cell 10
# ============================================================
# 🖥️  GRADIO FRONTEND — Sympriority Speech Input Panel
# Medical theme: white/light-grey bg · red + blue accents
# Uses whisper-medium (GPU-safe on free Colab T4)
# ✅ STANDALONE — no dependency on any previous cell
# ============================================================

# ── Step 0: Dependencies ─────────────────────────────────────
!apt-get install -y -q ffmpeg
!pip install -q openai-whisper "gradio>=4.0"

import torch, whisper, gradio as gr

# ── Step 1: Load model once ──────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {device.upper()}")
print("⏳ Loading whisper-medium ...")
asr_model = whisper.load_model("medium", device=device)
print("✅ Ready — launching UI ...\n")

# ── Step 2: Processing function ──────────────────────────────
# task="translate" → single pass: Whisper auto-detects language + outputs English
# fp16=False — avoids the silent CPU fallback bug in openai-whisper on Colab T4
#   (fp16=True with openai-whisper package causes dtype mismatch → runs on CPU → 3+ min)
def process_audio(audio_path):
    if audio_path is None:
        return "⚠️  No audio recorded. Press the microphone button, speak, then stop."
    try:
        result = asr_model.transcribe(
            audio_path,
            task="translate",
            beam_size=3,
            fp16=False          # keep False — prevents silent CPU fallback on T4
        )
        english = result["text"].strip()
        return english if english else "⚠️  Could not transcribe. Speak clearly and retry."
    except Exception as e:
        return f"❌ Error: {str(e)}"

# ── Step 3: Custom medical CSS ───────────────────────────────
# Gradio 4.x wraps components in shadow containers.
# We target both the elem_id and Gradio's internal .wrap / .block classes
# to ensure overrides work regardless of Gradio minor version.
medical_css = """
/* ── Page & container background ── */
body, .gradio-container, .gradio-container > .main,
.gradio-container .prose { background-color: #f0f4f8 !important; }

/* ── Audio recorder card ── */
#mic-box, #mic-box .wrap, #mic-box > div {
    background: #ffffff !important;
    border: 2px solid #1a5276 !important;
    border-radius: 10px !important;
}

/* ── Submit button ── */
#submit-btn, #submit-btn button {
    background: #c0392b !important;
    color: #ffffff !important;
    border: none !important;
    border-radius: 8px !important;
    font-size: 1rem !important;
    font-weight: 700 !important;
    width: 100% !important;
    padding: 14px !important;
    cursor: pointer !important;
}
#submit-btn button:hover { background: #a93226 !important; }

/* ── Output textbox — target all Gradio wrapper layers ── */
#output-box,
#output-box .wrap,
#output-box .block,
#output-box > div,
#output-box textarea,
#output-box [data-testid="textbox"] {
    background: #ffffff !important;
    color: #1a252f !important;
}
#output-box textarea {
    border: 2px solid #1a5276 !important;
    border-radius: 8px !important;
    font-size: 1.05rem !important;
    line-height: 1.6 !important;
    padding: 12px !important;
    min-height: 120px !important;
}
#output-box label, #output-box span {
    color: #1a5276 !important;
    font-weight: 600 !important;
    background: transparent !important;
}

/* ── Footer ── */
#footer-note {
    color: #5d6d7e !important;
    font-size: 0.82rem !important;
    text-align: center !important;
    margin-top: 6px !important;
    background: transparent !important;
}
"""

# ── Step 4: Build UI ─────────────────────────────────────────
with gr.Blocks(css=medical_css, title="Sympriority — Voice Input") as demo:

    # Use gr.HTML with inline styles for the header.
    # gr.Column adds a white card wrapper that overrides any CSS background —
    # inline styles on a raw <div> are immune to Gradio's theming.
    gr.HTML("""
        <div style="
            background: linear-gradient(135deg, #c0392b 0%, #1a5276 100%);
            border-radius: 12px;
            padding: 22px 28px;
            margin-bottom: 14px;
        ">
            <h1 style="
                color: #ffffff;
                margin: 0 0 6px 0;
                font-size: 1.75rem;
                font-family: 'Segoe UI', Arial, sans-serif;
                font-weight: 700;
            ">🏥 Sympriority — Patient Voice Input</h1>
            <p style="
                color: #dce9f5;
                margin: 0;
                font-size: 0.95rem;
                font-family: 'Segoe UI', Arial, sans-serif;
            ">Speak your symptoms in
                <b style="color:#ffffff;">English, Hindi, Telugu, or Tamil</b>.
                The system translates everything to English automatically.
            </p>
        </div>
    """)

    audio_input = gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="🎤  Click the mic, speak your symptoms, then click Stop",
        elem_id="mic-box"
    )

    submit_btn = gr.Button(
        "▶  Analyse Speech",
        elem_id="submit-btn",
        size="lg"
    )

    # interactive=True (default) renders as a normal editable textarea —
    # Gradio applies light styling to it, making our CSS override reliable.
    english_out = gr.Textbox(
        label="🎯  Patient Symptom Description (English)",
        lines=5,
        placeholder="Translated English text will appear here after you click Analyse Speech ...",
        elem_id="output-box"
    )

    gr.HTML('<p id="footer-note">Model: openai/whisper-medium &nbsp;·&nbsp; '
            '~307M parameters &nbsp;·&nbsp; Supports EN · HI · TE · TA</p>')

    submit_btn.click(
        fn=process_audio,
        inputs=[audio_input],
        outputs=[english_out]
    )

# share=True creates a public gradio.live link — required in Colab
demo.launch(share=True)


In [ ]:
#cell 11
# ============================================================
# 🏥 SYMPRIORITY — FULL TRIAGE PIPELINE
#   Stage 1 : Audio → English text  (Whisper-medium, local GPU)
#   Stage 2 : Symptoms → Triage JSON (Gemma-2-2b-it, HF API)
# ✅ STANDALONE — safe to run alone OR after cells 9/10
# ✅ VRAM-smart — reuses Whisper if already loaded in session
# ============================================================

# ── Step 0: Dependencies ─────────────────────────────────────
!apt-get install -y -q ffmpeg
!pip install -q openai-whisper "gradio>=4.0" "huggingface_hub>=0.20" transformers accelerate

import torch, whisper, gradio as gr, json, re, os
from transformers import pipeline as hf_pipeline

# ── Step 1: HF API key ───────────────────────────────────────
HF_TOKEN = "Use hugging face token here"   # ----Add hugging face token
os.environ["HF_TOKEN"] = HF_TOKEN

# ── Step 2: Load Whisper — reuse if already in session ───────
# Cells 9 and 10 both load whisper-medium (~2 GB each).
# If either has already run, `asr_model` exists in global scope.
# Reusing it avoids loading a duplicate and saves ~2 GB VRAM.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {device.upper()}")

if "asr_model" not in dir() or asr_model is None:
    print("⏳ Loading whisper-medium (not found in session) ...")
    asr_model = whisper.load_model("medium", device=device)
    print("✅ Whisper-medium loaded")
else:
    print("✅ Whisper-medium reused from previous cell (no extra VRAM used)")

# Local triage model — downloads weights from HF, runs on Colab GPU
# Llama-3.2-1B-Instruct: 1B params, ~2GB VRAM, no API routing involved.
# Requires accepting Meta Llama 3.2 license at:
#   huggingface.co/meta-llama/Llama-3.2-1B-Instruct
print("⏳ Loading Llama-3.2-1B-Instruct (local GPU) ...")
triage_pipe = hf_pipeline(
    "text-generation",
    model="meta-llama/Llama-3.2-1B-Instruct",
    token=HF_TOKEN,
    device_map="auto",
    torch_dtype=torch.float16
)
print("✅ Llama-3.2-1B-Instruct loaded\n⏳ Launching UI ...")

# ── Step 3: Triage prompt & LLM call ─────────────────────────
# Low temperature (0.2) ensures consistent, deterministic JSON output.
# The prompt instructs Gemma to return ONLY a JSON block — no prose.
TRIAGE_PROMPT = """You are a clinical triage AI for a hospital OPD. Assess patient symptom urgency accurately using the criteria below.

RISK LEVEL GUIDELINES — apply strictly:

CRITICAL (score 9-10): Immediately life-threatening. Emergency intervention required NOW.
  Examples: chest pain with sweating/arm pain, difficulty breathing or choking, stroke signs
  (face drooping, slurred speech, sudden arm weakness), unconsciousness, severe allergic
  reaction (throat swelling), uncontrolled bleeding, poisoning, suspected heart attack.

HIGH (score 6-8): Serious — needs doctor within 30-60 minutes.
  Examples: fever above 39.5°C/103°F, severe abdominal pain, head injury with confusion,
  suspected fracture, persistent vomiting with dehydration, child with high fever, chest
  tightness without other cardiac signs, signs of infection spreading.

MODERATE (score 3-5): Needs medical attention today, not an emergency.
  Examples: fever 38°C-39.5°C, ear/throat infection, urinary tract infection, mild-moderate
  abdominal pain, minor wounds needing stitches, back pain, persistent diarrhea,
  toothache, mild allergic rash.

LOW (score 1-2): Routine care, no urgency.
  Examples: common cold, mild headache without other symptoms, minor cuts or bruises,
  mild cough without fever, runny nose, routine medication refill, skin rash without
  swelling or spreading.

CRITICAL RULE: A simple cold, mild fever, or mild headache alone is ALWAYS Low or Moderate — NEVER Critical or High.
Assign scores honestly based on clinical evidence in the symptoms, not assumptions.

Respond ONLY with valid JSON — no text outside the JSON block:
{
  "risk_level": "Critical" | "High" | "Moderate" | "Low",
  "priority_score": <integer 1-10, 10 = most critical>,
  "recommended_action": "<specific action, e.g. 'Call emergency services immediately' or 'Visit OPD today' or 'Rest, fluids, OTC paracetamol'>",
  "reasoning": "<1-2 sentences referencing the specific symptoms and why this risk level applies>"
}"""

def run_triage(symptom_text: str) -> dict:
    messages = [
        {"role": "system", "content": TRIAGE_PROMPT},
        {"role": "user", "content": f"Patient symptoms:\n{symptom_text}"}
    ]
    result = triage_pipe(messages, max_new_tokens=300, do_sample=False, return_full_text=False)
    raw = result[0]["generated_text"].strip()
    if isinstance(raw, list):   # some versions return list of dicts
        raw = raw[-1]["content"].strip()

    # Extract JSON robustly — handles cases where model adds surrounding text
    match = re.search(r'\{[\s\S]*?\}', raw)
    if match:
        return json.loads(match.group())
    # Fallback if model returns unexpected format
    return {
        "risk_level": "Unknown",
        "priority_score": 0,
        "recommended_action": "Please review manually.",
        "reasoning": raw
    }

# ── Step 4: Full pipeline function ───────────────────────────
RISK_COLORS = {
    "Critical": "#c0392b",   # red
    "High":     "#e67e22",   # orange
    "Moderate": "#d4ac0d",   # amber
    "Low":      "#27ae60",   # green
    "Unknown":  "#7f8c8d"    # grey
}

def full_pipeline(audio_path):
    if audio_path is None:
        return "⚠️ No audio recorded. Click the mic button first.", ""

    # — Stage 1: Whisper transcription —
    try:
        r = asr_model.transcribe(
            audio_path,
            task="translate",   # auto-detects + translates to English in one pass
            beam_size=3,
            fp16=False          # prevents silent CPU fallback bug on T4
        )
        english = r["text"].strip()
    except Exception as e:
        return f"❌ Transcription error: {str(e)}", ""

    if not english:
        return "⚠️ Could not transcribe. Speak louder/closer and retry.", ""

    # — Stage 2: Gemma triage via HF API —
    try:
        triage = run_triage(english)
    except Exception as e:
        return english, f"<p style='color:red'>❌ Triage error: {str(e)}</p>"

    risk   = triage.get("risk_level", "Unknown")
    score  = triage.get("priority_score", 0)
    action = triage.get("recommended_action", "")
    reason = triage.get("reasoning", "")
    color  = RISK_COLORS.get(risk, "#7f8c8d")

    # Build color-coded triage card as HTML
    triage_html = f"""
    <div style="font-family:'Segoe UI',Arial,sans-serif; padding:18px;
                background:#ffffff; border-radius:10px;
                border: 2px solid {color}; margin-top:4px;">

        <div style="display:flex; align-items:center; gap:14px; margin-bottom:14px;">
            <div style="background:{color}; color:#fff; padding:8px 20px;
                        border-radius:6px; font-size:1.15rem; font-weight:700;
                        letter-spacing:0.5px;">
                {risk.upper()}
            </div>
            <div style="font-size:1rem; color:#1a252f;">
                Priority Score:
                <span style="color:{color}; font-weight:700; font-size:1.2rem;">
                    {score}<span style="font-size:0.85rem; color:#5d6d7e;">/10</span>
                </span>
            </div>
        </div>

        <div style="margin-bottom:10px; padding:10px; background:#f8f9fa;
                    border-radius:6px; border-left: 4px solid #1a5276;">
            <span style="color:#1a5276; font-weight:600; font-size:0.9rem;">
                📋 RECOMMENDED ACTION
            </span>
            <p style="margin:6px 0 0 0; color:#1a252f; font-size:0.97rem;">{action}</p>
        </div>

        <div style="padding:10px; background:#f8f9fa; border-radius:6px;
                    border-left: 4px solid #7f8c8d;">
            <span style="color:#1a5276; font-weight:600; font-size:0.9rem;">
                🧠 CLINICAL REASONING
            </span>
            <p style="margin:6px 0 0 0; color:#5d6d7e; font-size:0.93rem;">{reason}</p>
        </div>
    </div>
    """
    return english, triage_html

# ── Step 5: Gradio UI ─────────────────────────────────────────
triage_css = """
body, .gradio-container, .gradio-container > .main,
.gradio-container .prose { background-color: #f0f4f8 !important; }

#mic-box2, #mic-box2 .wrap, #mic-box2 > div {
    background: #ffffff !important;
    border: 2px solid #1a5276 !important;
    border-radius: 10px !important;
}
#run-btn, #run-btn button {
    background: #c0392b !important; color: #ffffff !important;
    border: none !important; border-radius: 8px !important;
    font-size: 1rem !important; font-weight: 700 !important;
    width: 100% !important; padding: 14px !important;
}
#run-btn button:hover { background: #a93226 !important; }

#eng-out, #eng-out .wrap, #eng-out > div,
#eng-out textarea, #eng-out [data-testid="textbox"] {
    background: #ffffff !important; color: #1a252f !important;
}
#eng-out textarea {
    border: 2px solid #1a5276 !important; border-radius: 8px !important;
    font-size: 1rem !important; line-height: 1.6 !important; padding: 12px !important;
}
#eng-out label, #eng-out span { color: #1a5276 !important; font-weight:600 !important; background:transparent !important; }
"""

with gr.Blocks(css=triage_css, title="Sympriority — Triage") as triage_demo:

    gr.HTML("""
        <div style="background:linear-gradient(135deg,#c0392b 0%,#1a5276 100%);
                    border-radius:12px; padding:22px 28px; margin-bottom:14px;">
            <h1 style="color:#fff; margin:0 0 6px 0; font-size:1.75rem;
                       font-family:'Segoe UI',Arial,sans-serif; font-weight:700;">
                🏥 Sympriority — Patient Triage
            </h1>
            <p style="color:#dce9f5; margin:0; font-size:0.95rem;
                      font-family:'Segoe UI',Arial,sans-serif;">
                Speak your symptoms in
                <b style="color:#fff;">English, Hindi, Telugu, or Tamil</b>.
                AI assesses risk and recommends action.
            </p>
        </div>
    """)

    audio_in = gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="🎤  Click mic → speak symptoms → click Stop",
        elem_id="mic-box2"
    )

    run_btn = gr.Button("▶  Transcribe & Analyse", elem_id="run-btn", size="lg")

    english_out = gr.Textbox(
        label="📝  Symptom Description (English)",
        lines=3,
        placeholder="Transcribed English text will appear here ...",
        elem_id="eng-out"
    )

    triage_out = gr.HTML(
        label="🩺  Triage Assessment",
        value="<p style='color:#7f8c8d; font-family:Segoe UI,Arial,sans-serif; "
              "padding:12px;'>Triage result will appear here after analysis ...</p>"
    )

    gr.HTML(
        "<p style='color:#5d6d7e; font-size:0.82rem; text-align:center; margin-top:6px;'>"
        "Stage 1: openai/whisper-medium &nbsp;·&nbsp; "
        "Stage 2: meta-llama/Llama-3.2-1B-Instruct (local GPU)"
        "</p>"
    )

    run_btn.click(
        fn=full_pipeline,
        inputs=[audio_in],
        outputs=[english_out, triage_out]
    )

triage_demo.launch(share=True)


In [ ]:
#cell 12
# ============================================================
# 🧪 MINISTRAL-3B TEST — Quick standalone trial
#   Loads mistralai/Ministral-3B-Instruct locally on GPU
#   and runs a single triage query to see raw model output.
#   ✅ Does NOT affect cell 11 — completely independent
#   ✅ VRAM-safe — fits alongside Whisper on free T4
# ============================================================

# ── Dependencies (already installed if cell 11 ran) ──────────
!pip install -q transformers accelerate

import torch
from transformers import pipeline as hf_pipeline

# ── Load Ministral-3B locally ─────────────────────────────────
# ~6 GB VRAM in float16 — safe to run alongside Whisper (~2 GB).
# Total: ~8 GB — well within the free T4's 15 GB.
# Requires accepting Ministral license at:
#   huggingface.co/mistralai/Ministral-3B-Instruct
MINISTRAL_TOKEN = "Use hugging face token here"   # ----Add hugging face token

print("⏳ Loading Ministral-3B-Instruct ...")
ministral_pipe = hf_pipeline(
    "text-generation",
    model="mistralai/Ministral-3-3B-Instruct-2512",
    token=MINISTRAL_TOKEN,
    device_map="auto",
    torch_dtype=torch.float16
)
print("✅ Ministral-3B loaded")

# ── Test prompt ───────────────────────────────────────────────
TEST_SYMPTOMS = "I have a mild fever and a slight headache since this morning."

MISTRAL_PROMPT = """You are a clinical triage AI. Given patient symptoms, respond ONLY with valid JSON:
{
  "risk_level": "Critical" | "High" | "Moderate" | "Low",
  "priority_score": <integer 1-10>,
  "recommended_action": "<specific action>",
  "reasoning": "<1-2 sentences>"
}"""

messages = [
    {"role": "user", "content": f"{MISTRAL_PROMPT}\n\nPatient symptoms:\n{TEST_SYMPTOMS}"}
]

print(f"\n📋 Input: {TEST_SYMPTOMS}\n")
result = ministral_pipe(messages, max_new_tokens=300, do_sample=False, return_full_text=False)
raw = result[0]["generated_text"]
if isinstance(raw, list):
    raw = raw[-1]["content"]
print("🤖 Ministral-3B output:\n", raw.strip())